# Módulo 4: AgentCore Memory -- Contexto Persistente

![Overview](../shared/img/04.drawio.png)

Neste módulo, você vai configurar a **memória** da Aria, permitindo que ela se lembre de quem você é ao longo de várias conversas.

---

### O que você vai aprender

| Tópico | Detalhes |
|---|---|
| **Serviço AgentCore Memory** | A camada gerenciada de memória nativa do Bedrock AgentCore |
| **Memória de curto vs. longo prazo** | Como o STM (buffer de conversa) e o LTM (conhecimento extraído) operam em conjunto |
| **Estratégias de extração LTM** | Resumos de sessão, preferências do usuário e fatos semânticos |
| **Integração com Strands** | Como o `AgentCoreMemorySessionManager` se conecta em um agente Strands |

## Atualização de Dependências

A célula abaixo valida se todos os recursos dos módulos anteriores estão ativos na AWS. Se você pulou alguma etapa, o script fará o provisionamento automaticamente.

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared.ensure_ready import ensure_ready

# Verifica se o Runtime está rodando (necessário para este módulo).
config = ensure_ready("04")

---

## Entendendo a Memória

O AgentCore Memory concede ao seu agente a capacidade de **lembrar** informações entre as sessões. Ele opera em duas camadas:

### Memória de curto prazo (STM)

Quando o agente envia eventos da conversa para o AgentCore Memory, o serviço os armazena como **eventos**. Eles capturam cada interação. O agente pode buscar esses eventos para reidratar o histórico da conversa — útil para retomar uma sessão posteriormente. Enviar eventos para o Memory também é o que ativa a extração das memórias de longo prazo.

### Memória de longo prazo (LTM)

Conhecimento extraído que **persiste entre as sessões**. À medida que os eventos são enviados, o serviço executa estratégias de extração em segundo plano (background) para transformar o diálogo em conhecimento durável. Essa extração leva cerca de um minuto. O objetivo principal é fornecer contexto para as **futuras sessões**.

O AgentCore Memory suporta três estratégias nativas de extração LTM:

| Estratégia | Classe | Namespace | Propósito |
|---|---|---|---|
| **Session Summarizer** | `summaryMemoryStrategy` | `/summaries/{actorId}/{sessionId}` | Cria um resumo conciso da sessão para ser recuperado rapidamente no futuro |
| **Preference Learner** | `userPreferenceMemoryStrategy` | `/preferences/{actorId}` | Detecta e armazena preferências do usuário (ex: 'Prefiro Python a Java') |
| **Fact Extractor** | `semanticMemoryStrategy` | `/facts/{actorId}` | Extrai informações factuais (ex: 'Trabalho na Acme Corp', 'Meu nome é Alex') |

Cada estratégia grava em seu próprio **namespace**, e o `{actorId}` é substituído dinamicamente em tempo de execução pela identidade do usuário autenticado, mantendo as memórias de cada usuário totalmente isoladas.

> **Documentation:** [AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)

---

### Criando o recurso Memory na AWS

A API `create_memory` provisiona um **recurso de memória** — pense nisso como a criação do repositório onde as memórias serão armazenadas. Este recurso é um banco de dados gerenciado com base no DynamoDB.

Você _pode_ configurar um recurso sem nenhuma estratégia LTM. Nesse caso, ele apenas armazena os eventos do chat. As estratégias LTM são complementos opcionais que extraem o conhecimento orgânico.

Aqui configuramos as três estratégias para a Aria ter a experiência completa:
- **SessionSummarizer** — Resume as conversas quando a sessão é encerrada
- **PreferenceLearner** — Extrai preferências corporativas ('Prefiro modo escuro')
- **FactExtractor** — Armazena fatos semânticos puros ('Alex é engenheiro de software')

Cada estratégia possui:
- Um **name** (nome) e **description** (descrição) para identificação
- **Namespaces** — caminhos de template que delimitam o escopo das memórias (por usuário ou por sessão)

In [ ]:
import boto3
from botocore.exceptions import ClientError

region = "us-east-1"
client = boto3.client("bedrock-agentcore-control", region_name=region)

try:
    # Cria o recurso AriaMemory com as estratégias: SessionSummarizer, PreferenceLearner, FactExtractor.
    response = client.create_memory(
        name="AriaMemory",
        description="Memory for Aria personal assistant — conversation persistence and long-term user knowledge",
        eventExpiryDuration=90,  # Days before memory events expire
        memoryStrategies=[
            {
                "summaryMemoryStrategy": {
                    "name": "SessionSummarizer",
                    "description": "Summarizes conversation sessions for quick context retrieval",
                    "namespaces": ["/summaries/{actorId}/{sessionId}"],
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "PreferenceLearner",
                    "description": "Learns and stores user preferences across sessions",
                    "namespaces": ["/preferences/{actorId}"],
                }
            },
            {
                "semanticMemoryStrategy": {
                    "name": "FactExtractor",
                    "description": "Extracts and stores factual information from conversations",
                    "namespaces": ["/facts/{actorId}"],
                }
            },
        ],
    )
    memory_id = response["memory"]["id"]
    print(f"Memory created: {memory_id}")
    print(f"   Status: {response['memory'].get('status', 'CREATING')}")

except ClientError as e:
    if "already exists" in str(e):
        print("AriaMemory already exists -- looking it up...")
        paginator = client.get_paginator("list_memories")
        for page in paginator.paginate():
            for mem in page.get("memories", []):
                if mem["id"].startswith("AriaMemory"):
                    memory_id = mem["id"]
                    print(f"Found existing memory: {memory_id}")
                    break
    else:
        raise

### Aguardando o status ACTIVE do Memory

A criação da memória é assíncrona. Fazemos polling da API até o status ficar `ACTIVE`.

In [ ]:
import time

print(f"Waiting for memory {memory_id} to become ACTIVE...")
for i in range(30):
    resp = client.get_memory(memoryId=memory_id)
    memory = resp.get("memory", resp)
    status = memory.get("status", "UNKNOWN")
    print(f"  [{i*10}s] Status: {status}")
    if status == "ACTIVE":
        print("Memory is ACTIVE!")
        break
    if status in ("FAILED", "DELETE_FAILED"):
        print(f"Memory creation failed: {status}")
        break
    time.sleep(10)

### Salvando o ID do Memory

Persistimos o ID do Memory no diretório local de configuração para que o `ensure_ready()` e os scripts de deploy o encontrem nos módulos seguintes.

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import utils

utils.save_config("memory", {"memory_id": memory_id, "region": region})
print(f"Saved memory_id={memory_id}")

---
## Enable Tracing for Memory

Enable **Tracing** on the Memory resource so that memory operations (store, retrieve, extraction) appear in the AgentCore Observability dashboard.

1. Open the **Amazon Bedrock AgentCore** console and navigate to **Memory**
2. Select the **AriaMemory** resource
3. Scroll down to the **Tracing** section and click **Edit**

![Tracing section](../shared/img/tracing-01.png)

4. Toggle **Enable** on and click **Save**

![Enable tracing](../shared/img/tracing-02.png)

To see the full agent code, open [agent/main.py](agent/main.py) in a new tab.

---

## Deploy com o módulo Memory ativado

Vamos fazer o deploy da Aria com a variável de ambiente `MEMORY_ID` apontando para a Memória que acabamos de provisionar. O Runtime carregará o novo código e a nova configuração.

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared.deploy_agent import deploy

# Faz o deploy da V3 do agente (agora com memória persistente entre sessões).
result = deploy(
    agent_dir="agent",
    # ID do recurso de memória — passado como variável de ambiente para o Runtime.
    env_vars={"MEMORY_ID": memory_id},
)
runtime_arn = result["runtime_arn"]

---

## Testando a Memória: Conversa 1

Vamos ter uma conversa onde passaremos alguns fatos para a Aria. Ambas as mensagens usam a **mesma sessão (session ID)**, compartilhando o buffer de curto prazo (STM).

### Fale sobre você para a Aria

In [ ]:
import boto3, json, uuid
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import utils

data_client = boto3.client("bedrock-agentcore", region_name="us-east-1")

# Session 1: Tell Aria about yourself
session_1 = str(uuid.uuid4())
print(f"Session 1: {session_1[:16]}...")

response = data_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=session_1,
    payload=json.dumps({
        "prompt": "My name is Alex and I'm a software engineer. I prefer Python over Java.",
        # ID da sessão — a memória agrupa conversas por sessão e por usuário.
        "session_id": session_1,
    }).encode(),
)
utils.stream_sse_response(response["response"])

### Mesma sessão -- Recuperação de STM (curto prazo)

Aria deve conseguir lembrar seu nome, já que a primeira mensagem ainda está ativa na sessão da microVM.

In [ ]:
# Same session: STM recall (conversation buffer)
response = data_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=session_1,
    payload=json.dumps({
        "prompt": "What's my name and what language do I prefer?",
        # ID da sessão — a memória agrupa conversas por sessão e por usuário.
        "session_id": session_1,
    }).encode(),
)
utils.stream_sse_response(response["response"])

---

## Testando a Memória: Nova sessão (Longo Prazo)

Agora iniciaremos uma **nova sessão**. O buffer de conversa estará limpo, então a Aria só vai se lembrar dos seus dados se o LTM (longo prazo) tiver extraído os fatos com sucesso na AWS.

> **Nota:** A extração para o LTM roda em background (segundo plano). Geralmente leva cerca de 1 minuto. Se a nova sessão falhar em lembrar de você de primeira, aguarde 60 segundos e rode a célula novamente.

In [ ]:
# NEW session: LTM recall (cross-session)
session_2 = str(uuid.uuid4())
print(f"Session 2 (new): {session_2[:16]}...")

response = data_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=session_2,
    payload=json.dumps({
        "prompt": "What do you remember about me?",
        # ID da sessão — a memória agrupa conversas por sessão e por usuário.
        "session_id": session_2,
    }).encode(),
)
utils.stream_sse_response(response["response"])

Se a arquitetura estiver correta, a Aria lembrará do seu nome, sua profissão e sua preferência por Python — mesmo em um chat novo.

---

## Como a arquitetura funciona

Aqui está o fluxo operacional da arquitetura:

- A sessão é iniciada. O AgentCoreMemorySessionManager busca os eventos na AWS (se já existirem).
- O usuário digita a mensagem.
- O AgentCoreMemorySessionManager faz uma query no LTM e retorna memórias ligadas semanticamente, injetando esse contexto no loop do agente.
- O Agente processa a inferência.
- Todas as interações são disparadas como eventos na AWS via AgentCoreMemorySessionManager.
- A AWS (AgentCore Memory) executa a rotina de extração em background para destilar as informações duráveis nos namespaces.

As configurações de threshold (limite de busca semântica) são diferentes:

- **Preferências** (`relevance_score=0.7`): Exige alta relevância de busca.
- **Fatos** (`relevance_score=0.3`, `top_k=10`): Busca mais abrangente para compor o contexto geral.
- **Resumos** (`relevance_score=0.5`): Um equilíbrio entre os dois pólos.

> **Go deeper:** [AgentCore Memory documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html) — includes custom memory strategies, memory management APIs, and advanced retrieval configuration

---

## O que teremos no próximo lab?

A Memória deu um cérebro duradouro para a Aria. Mas no momento, ela só sabe **conversar**. Ela não consegue manipular sistemas externos por você.

No **Módulo 5: Gateway e Identidade**, vamos conectar a Aria a uma API de Tarefas através do AgentCore Gateway e aplicar proteção JWT para garantir o isolamento da identidade corporativa.

---

## Progress

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared.progress import show

show("04")

---

**Next up: [Module 5 -- Connect Aria to the Gateway](../05-gateway-identity/notebook.ipynb)**